# IT Service Management (ITSM)

## Introduction

ABC Tech is a mid-sized IT-enabled organization that has been operating for over a decade, handling an average of 22,000–25,000 IT incidents and tickets every month. The organization follows the ITIL (Information Technology Infrastructure Library) framework, managing its operations through structured incident management, problem management, change management, and configuration management processes.

While these ITIL practices have matured over the years, a recent internal audit revealed that further process-level improvements are unlikely to yield significant returns. At the same time, a customer satisfaction survey highlighted incident management as one of the weakest areas of service delivery, prompting the management to explore new avenues for improvement.

Recognizing the growing potential of Machine Learning (ML) in transforming ITSM (IT Service Management) operations, ABC Tech identified four key areas where predictive analytics could add value: forecasting high-priority incidents, projecting future ticket volumes, automating ticket tagging for faster routing, and predicting potential failures related to change requests (RFCs).

This project focuses on the first of these objectives — predicting high-priority IT incidents (Priority 1 and 2 tickets) using historical ticket data spanning 2012–2014, sourced from ABC Tech's internal ITSM database. The goal is to build a Machine Learning model capable of identifying high-priority incidents in advance, enabling the support team to take preventive action, reduce resolution delays, and ultimately improve overall service quality and customer satisfaction.

## Problem Statement

### Current Situation
ABC Tech, a mid-sized IT-enabled organization, handles approximately 22,000–25,000 IT incidents every month through its ITIL-based service management process.

### Core Issue
- Despite following mature ITIL practices, recent customer feedback has rated the incident management process **poorly**.
- An internal audit confirmed that traditional process-improvement methods have reached a point of **diminishing returns**.
- High-priority incidents (Priority 1 and 2) are currently identified **only after they occur** — the process is entirely reactive.

### Impact of the Problem
- Delayed response to critical incidents.
- Increased service disruption for end users.
- Inefficient allocation of support resources.
- Declining customer satisfaction scores.

### Objective
To leverage historical incident data (2012–2014) and build a **Machine Learning classification model** that can predict whether an incoming IT ticket is likely to be a **high-priority incident**, using features such as:
- Impact
- Urgency
- Configuration Item (CI) category
- Reassignment history

### Expected Outcome
By enabling early identification of high-priority tickets, ABC Tech aims to:
- Reduce incident response time.
- Minimize service disruption.
- Improve overall efficiency and reliability of the IT incident management process.

In [35]:
import pandas as pd

In [36]:
df = pd.read_csv("../data/row_data/itsm.csv")
df.head()

C:\Users\rashi\AppData\Local\Temp\ipykernel_18424\2561833240.py:1: DtypeWarning: Columns (0: Urgency) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/row_data/itsm.csv")


,CI_Name,CI_Cat,CI_Subcat,WBS,Incident_ID,Status,Impact,Urgency,Priority,number_cnt,...,Reopen_Time,Resolved_Time,Close_Time,Handle_Time_hrs,Closure_Code,No_of_Related_Interactions,Related_Interaction,No_of_Related_Incidents,No_of_Related_Changes,Related_Change
0,SUB000508,subapplication,Web Based Application,WBS000162,IM0000004,Closed,4,4,4.0,0.601292,...,NaN,04-11-2013 13:50,04-11-2013 13:51,"3,87,16,91,111",Other,1.0,SD0000007,2.0,NaN,NaN
1,WBA000124,application,Web Based Application,WBS000088,IM0000005,Closed,3,3,3.0,0.415050,...,02-12-2013 12:31,02-12-2013 12:36,02-12-2013 12:36,"4,35,47,86,389",Software,1.0,SD0000011,1.0,NaN,NaN
2,DTA000024,application,Desktop Application,WBS000092,IM0000006,Closed,NS,3,NaN,0.517551,...,NaN,13-01-2014 15:12,13-01-2014 15:13,"4,84,31,19,444",No error - works as designed,1.0,SD0000017,NaN,NaN,NaN
3,WBA000124,application,Web Based Application,WBS000088,IM0000011,Closed,4,4,4.0,0.642927,...,NaN,14-11-2013 09:31,14-11-2013 09:31,"4,32,18,33,333",Operator error,1.0,SD0000025,NaN,NaN,NaN
4,WBA000124,application,Web Based Application,WBS000088,IM0000012,Closed,4,4,4.0,0.345258,...,NaN,08-11-2013 13:55,08-11-2013 13:55,"3,38,39,03,333",Other,1.0,SD0000029,NaN,NaN,NaN


## Data Understamding

In [37]:
df.shape

(46606, 25)

In [38]:
df.size

1165150

In [39]:
df.dtypes

CI_Name                           str
CI_Cat                            str
CI_Subcat                         str
WBS                               str
Incident_ID                       str
Status                            str
Impact                            str
Urgency                        object
Priority                      float64
number_cnt                    float64
Category                          str
KB_number                         str
Alert_Status                      str
No_of_Reassignments           float64
Open_Time                         str
Reopen_Time                       str
Resolved_Time                     str
Close_Time                        str
Handle_Time_hrs                   str
Closure_Code                      str
No_of_Related_Interactions    float64
Related_Interaction               str
No_of_Related_Incidents       float64
No_of_Related_Changes         float64
Related_Change                    str
dtype: object

In [40]:
df.nunique()

CI_Name                        3019
CI_Cat                           12
CI_Subcat                        64
WBS                             274
Incident_ID                   46606
Status                            2
Impact                            6
Urgency                          11
Priority                          5
number_cnt                    46606
Category                          4
KB_number                      1825
Alert_Status                      1
No_of_Reassignments              41
Open_Time                     34636
Reopen_Time                    2244
Resolved_Time                 33627
Close_Time                    34528
Handle_Time_hrs               30638
Closure_Code                     14
No_of_Related_Interactions       49
Related_Interaction           43060
No_of_Related_Incidents          24
No_of_Related_Changes             4
Related_Change                  232
dtype: int64

In [41]:
df.isnull().sum()

CI_Name                           0
CI_Cat                          111
CI_Subcat                       111
WBS                               0
Incident_ID                       0
Status                            0
Impact                            0
Urgency                           0
Priority                       1380
number_cnt                        0
Category                          0
KB_number                         0
Alert_Status                      0
No_of_Reassignments               1
Open_Time                         0
Reopen_Time                   44322
Resolved_Time                  1780
Close_Time                        0
Handle_Time_hrs                   1
Closure_Code                    460
No_of_Related_Interactions      114
Related_Interaction               0
No_of_Related_Incidents       45384
No_of_Related_Changes         46046
Related_Change                46046
dtype: int64

In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 46606 entries, 0 to 46605
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   CI_Name                     46606 non-null  str    
 1   CI_Cat                      46495 non-null  str    
 2   CI_Subcat                   46495 non-null  str    
 3   WBS                         46606 non-null  str    
 4   Incident_ID                 46606 non-null  str    
 5   Status                      46606 non-null  str    
 6   Impact                      46606 non-null  str    
 7   Urgency                     46606 non-null  object 
 8   Priority                    45226 non-null  float64
 9   number_cnt                  46606 non-null  float64
 10  Category                    46606 non-null  str    
 11  KB_number                   46606 non-null  str    
 12  Alert_Status                46606 non-null  str    
 13  No_of_Reassignments         46605 non-null

In [43]:
df.describe()

,Priority,number_cnt,No_of_Reassignments,No_of_Related_Interactions,No_of_Related_Incidents,No_of_Related_Changes
count,45226.000000,46606.000000,46605.000000,46492.000000,1222.000000,560.000000
mean,4.215805,0.499658,1.131831,1.149897,1.669394,1.058929
std,0.705624,0.288634,2.269774,2.556338,3.339687,0.403596
min,1.000000,0.000023,0.000000,1.000000,1.000000,1.000000
25%,4.000000,0.248213,0.000000,1.000000,1.000000,1.000000
50%,4.000000,0.500269,0.000000,1.000000,1.000000,1.000000
75%,5.000000,0.749094,2.000000,1.000000,1.000000,1.000000
max,5.000000,0.999997,46.000000,370.000000,63.000000,9.000000


In [44]:
df.describe(include='object')

C:\Users\rashi\AppData\Local\Temp\ipykernel_18424\87514550.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object')


,CI_Name,CI_Cat,CI_Subcat,WBS,Incident_ID,Status,Impact,Urgency,Category,KB_number,Alert_Status,Open_Time,Reopen_Time,Resolved_Time,Close_Time,Handle_Time_hrs,Closure_Code,Related_Interaction,Related_Change
count,46606,46495,46495,46606,46606,46606,46606,46606,46606,46606,46606,46606,2284,44826,46606,46605,46146,46606,560
unique,3019,12,64,274,46606,2,6,11,4,1825,1,34636,2244,33627,34528,30638,14,43060,232
top,SUB000456,application,Server Based Application,WBS000073,IM0000004,Closed,4,4,incident,KM0001106,closed,24-03-2014 08:54,15-10-2013 09:53,10-10-2013 12:53,02-10-2013 15:20,0,Other,#MULTIVALUE,C00003013
freq,3050,32900,18811,13342,1,46597,22556,15526,37748,1106,46606,7,2,7,21,236,16470,3434,110


In [45]:
df.duplicated().sum()

np.int64(0)

### Source
The dataset was extracted from a MySQL database (`project_itsm`) hosted on a remote server, accessed using read-only credentials provided as part of the project.

### Method
- Connected to the database using **MySQL Workbench** (Standard TCP/IP connection).
- Queried the full incident table using `SELECT * FROM table_name;`.
- Exported the query result set to a CSV file for local processing.

### Challenges Faced

| Issue | Cause | Resolution |
|---|---|---|
| Only 1,000 rows exported initially | MySQL Workbench's default "Limit Rows" setting | Increased `Limit Rows Count` to 100,000 in **Preferences → SQL Editor → SQL Execution** |
| `ParserError: EOF inside string` while loading CSV in pandas | Malformed quote characters in some text fields | Loaded data using `engine='python', quoting=3, on_bad_lines='skip'` |

### Final Extracted Dataset
- **Total records:** 46,606
- **Total columns:** 25
- **Verification:** Row count confirmed using `SELECT COUNT(*)` query, matching the project documentation's stated dataset size (~46,000 records from 2012–2014).
- **File saved as:** `data/raw/itsm.csv`

### Note
Data quality issues identified in columns such as `Urgency`, `Impact`, and `Handle_Time_hrs` (inconsistent formatting) were addressed separately in the **Data Cleaning** phase.